# =============================================================================
# 論文：以機器學習為基礎之通用型組織績效評核偏誤自動化偵測框架
# =============================================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
 
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_predict, KFold, GroupKFold
from sklearn.metrics import r2_score
from scipy import stats

# =============================================================================
模組 1：HRSchemaMapper (資料對應與清洗層)
這個模組負責把資料整理成系統看得懂的格式，是此平台的守門員。
(1)__init__ (初始化)：接收網頁前端傳來的設定，例如 HR 指定了哪些欄位是目標、哪些是敏感特徵，並把它們記下來。
(2)validate_and_clean (驗證與清洗)：
    a.檢查缺漏：確認 HR 選的欄位在 CSV 裡是不是真的存在。
    b.統一敏感特徵格式：把性別等文字全部轉小寫、去除空白、把簡寫（如 m、f）統一轉成 male、female。
    c.自動補值 (Imputation)：機器學習不允許有空值 (NA)，因此針對數值型資料填補「中位數」（避免極端值拉偏），對文字型資料填補「unknown」。
# =============================================================================

In [ ]:
# ==========================================
# 模組 1：Schema Mapping 與資料驗證層
# ==========================================
class HRSchemaMapper:
    def __init__(self, config):
        """
        config 範例：
        {
            "target"    : "ManagerRating",
            "sensitive" : ["Gender", "Ethnicity"],
            "objective" : ["YearsAtCompany", "X_EngagementRate", "Experience_SEM"]
        }
        """
        self.config    = config
        self.target    = config['target']
        self.sensitive = list(config.get('sensitive', []))   # 複製，不汙染原始 config
        self.objective = list(config.get('objective', []))   # 複製，不汙染原始 config

    def validate_and_clean(self, df):
        """驗證欄位存在、清洗髒資料、填補遺失值"""
        all_cols = [self.target] + self.sensitive + self.objective

        # a.檢查缺漏
        missing  = [c for c in all_cols if c not in df.columns]
        if missing:
            raise ValueError(f"❌ 上傳資料缺少欄位：{missing}")
 
        cleaned = df.copy()

        # 目標變數不可用中位數填補，否則會製造一批人工零偏誤列
        before = len(cleaned)
        cleaned = cleaned.dropna(subset=[self.target]).reset_index(drop=True)
        dropped = before - len(cleaned)
        if dropped > 0:
            print(f"⚠️  [資料驗證] 目標變數缺失，已丟棄 {dropped} 筆（不以中位數填補目標）。")

        # b.處理敏感特徵的髒資料 (例如: Male, male, M 統一化)
        for col in self.sensitive:
            if not pd.api.types.is_numeric_dtype(cleaned[col]):  # 用 is_numeric_dtype 反向判斷，避免新版 pandas 字串欄位 dtype
                cleaned[col] = (cleaned[col].astype(str)  
                                            .str.lower()  # 轉小寫
                                            .str.strip()  # 去除頭尾空白
                                            .replace({'m': 'male', 'f': 'female'}))  # M/F 映射還原

        # c.自動填補遺失值 (Robust Imputation)：數值 → 中位數、類別 → 'unknown'（僅針對 objective 與 sensitive，不含 target）
        for col in self.objective + self.sensitive:
            if pd.api.types.is_numeric_dtype(cleaned[col]):
                cleaned[col] = cleaned[col].fillna(cleaned[col].median()) # 數值型用中位數填補，避免極端值干擾
            else:
                cleaned[col] = cleaned[col].fillna('unknown') # 類別型用 'Unknown' 填補
 
        print("✅ [資料驗證] Schema 對應與基礎清洗完成。")
        return cleaned

# =============================================================================
模組 2：AutoFeatureEngineer (自動特徵工程模組)
這個模組將不同單位的數字轉化為影響/準確度，是此平台的翻譯官與強化器。
_fitted = False：確保資料的「標準」只在第一次訓練時被建立，後續重複呼叫時不會被覆蓋，防止重複 fit。
fit_transform (轉換特徵)：
    a.Label Encoding (標籤編碼)：把文字類的客觀特徵（如部門、學歷）轉換成數字（如 0, 1, 2...），讓模型讀得懂。
    b.Standard Scaling (標準化)：把所有數值特徵（如年資、薪水）縮放到相同的比例。這樣模型才不會因為薪水數字很大（如 50000），年資數字很小（如 5），就誤以為薪水比較重要。
    c.產生交互特徵：自動把前兩個數值特徵「相乘」創造一個新特徵，幫助模型發現隱藏的非線性關係，提升預測準確度。
# =============================================================================

In [ ]:
# ==========================================
# 模組 2：自動特徵工程模組 
# ==========================================

class AutoFeatureEngineer:
    def __init__(self):
        self.scalers          = {}
        self.encoders         = {}
        self.numeric_cols     = []
        self.categorical_cols = []
        self._fitted          = False   # 防止重複 fit (fit：學，吸收資料、記憶參數、找出規律)

    def fit_transform(self, df, objective_cols):
        """
        自動判斷型態、Encoding、Scaling、進行特徵工程（產生交互特徵）
        """
        # ── 判斷型態並 fit ─────────────────────────────────
        if not self._fitted:
            for col in objective_cols:
                if pd.api.types.is_numeric_dtype(df[col]):
                    self.numeric_cols.append(col)
                else:
                    self.categorical_cols.append(col)
 
            for col in self.categorical_cols:
                le = LabelEncoder()
                le.fit(df[col].astype(str))
                self.encoders[col] = le
 
            if self.numeric_cols:
                self.scalers['main'] = StandardScaler()
                self.scalers['main'].fit(df[self.numeric_cols])
 
            self._fitted = True
 
        # ── Transform ─────────────────────────────────────
        processed = df.copy()
        new_cols  = list(objective_cols)  
 
        # (1)類別型特徵自動 Encoding（對未見過的類別做防呆）
        for col in self.categorical_cols:
            le = self.encoders[col]
            known = set(le.classes_)
            processed[col] = processed[col].astype(str).apply(
                lambda v: v if v in known else le.classes_[0]
            )
            processed[col] = le.transform(processed[col])

        # (2)數值型特徵自動 Scaling（隨機森林對尺度不敏感，標準化不影響預測；此處保留以與論文敘述一致，且讓未來若改用線性模型時仍適用）
        if self.numeric_cols:
            processed[self.numeric_cols] = self.scalers['main'].transform(
                processed[self.numeric_cols]
            )

        print("✅ [特徵工程] 自動型態判定與轉換完成。")
        return processed, new_cols
 

# =============================================================================
模組 3：GeneralizedBiasEngine (通用型偏誤偵測框架)
這個模組負責計算客觀能力與衡量偏誤。它包含了一個統一入口 (run) 和四個核心武器：

武器 1：_run_residual_bias (殘差偏誤分析)：抓出「個人層級」的偏誤（誰被打壓、誰被偏袒）
1.Cross-Validation (交叉驗證)：把員工分成 5 份，輪流用其中 4 份訓練，去預測剩下那 1 份的客觀分數，讓 Objective_Rating 具備最真實的客觀能力值。
2.計算偏誤 (Bias)：核心公式 實際主管評分 - 客觀預測評分。
3.動態門檻 (Sigma Threshold)：利用統計學的標準差（1.5σ），抓出偏誤值異常高或異常低的員工，貼上 Suggest Lower 或 Suggest Higher 的標籤。

武器 2：run_bias_tests (統計偏誤檢定)：證明抓出來的偏誤「在學術上是不是真的存在」
依照敏感特徵（如性別）分群，比較各群的偏誤平均值：只有 2 個群體（如男女）使用 T-test、有 3 個以上群體（如多個種族）使用 One-way ANOVA。
計算效果量 (Effect Size)：除了算出 p-value，此模型還計算了 Cohen's d 或 η²，說明「偏誤不僅存在，而且它的嚴重程度是小、中、還是大」。

武器 3：generate_shap_summary (SHAP 解釋圖)
打開 AI 的黑盒子，解釋模型是依據哪些特徵做決定的（例如：年資高加分、滿意度低扣分），讓此模型具備「可解釋性 AI (XAI)」的能力。

武器 4：_run_group_fairness (群體公平性分析)
檢視「公司整體的制度」是否有系統性歧視。
1.二元化：把連續的考績分數，依照門檻（預設前 30%）切分成「高績效者 (High Performer)」與一般人。
2.自動推斷基準組：系統會自動找出拿到高績效比例「最高」的那個群體（如白人男性），設為 100% 的基準 (Privileged Group)。
3.計算 DI 指標：計算其他弱勢族群拿高分的機率，除以基準組的機率。如果低於 0.8（80% 法則），系統就會判定該群體處於潛在的系統性弱勢。
# =============================================================================

In [ ]:
# ==========================================
# 模組 3：通用偏誤偵測框架 (Bias Detection Framework)
# ==========================================
class GeneralizedBiasEngine:
    def __init__(self, target_col, objective_cols, sensitive_cols):
        self.target_col     = target_col
        self.objective_cols = list(objective_cols)
        self.sensitive_cols = list(sensitive_cols)
        self.model          = None
        self.X_all          = None   # 全量特徵（供 SHAP 使用）
    
    # ── 對外入口 ────────────────────────────────────────────
    # 支援多種偏誤定義的統一介面：residual 回歸偏誤(殘差)、group_fairness 群體公平性  
    def run(self, df, method="residual", **kwargs):
        if method == "residual":
            return self._run_residual_bias(df, **kwargs)
        elif method == "group_fairness":
            return self._run_group_fairness(df, **kwargs)
        else:
            raise ValueError(f"❌ 不支援的分析方法：{method}")

    # ── 1.Residual Bias 殘差偏誤分析 (Actual - Predicted) ────
    def _run_residual_bias(self, df, sigma_threshold=1.5, group_col=None):
        """
        使用 Cross-Validation 避免資料洩漏：
        原始做法：model.fit(X, y) → model.predict(X)
          → 模型在自己的訓練資料上預測，幾乎完美（R²=0.72），Bias 幾乎都是 0 → 這不是「客觀能力值」，而是「過擬合的自我複製」
        修正做法：Cross-Validated Predictions
          → 每個員工的預測值，都來自沒有看過他的那一份模型，偏誤值才有意義
        group_col：若資料為一人多列，傳入員工 ID 欄位，改用 GroupKFold，確保同一員工不會同時出現在訓練與驗證 fold（杜絕個體層級洩漏）。
        """  
        X = df[self.objective_cols].copy()
        y = df[self.target_col].astype(float)
 
        # (1)真正的客觀能力值：使用交叉驗證產生客觀的預測值 cross-validated (cv) predictions（5-fold）
        rf = RandomForestRegressor(
            n_estimators=200, max_depth=10,
            min_samples_leaf=5, random_state=42, n_jobs=-1
        )
 
        # 依資料結構選擇交叉驗證策略
        if group_col is not None and group_col in df.columns:
            n_groups = df[group_col].nunique()
            n_splits = min(5, n_groups)
            gkf = GroupKFold(n_splits=n_splits)
            cv_preds = cross_val_predict(rf, X, y, cv=gkf, groups=df[group_col])
            cv_note = f"GroupKFold(by={group_col}, k={n_splits})"
        else:
            cv = KFold(n_splits=5, shuffle=True, random_state=42)
            cv_preds = cross_val_predict(rf, X, y, cv=cv)
            cv_note = "KFold(k=5)"

        # (2)訓練一個最終全量模型，專門供後續 SHAP 解釋用（沒有使用於計算 Bias）
        rf.fit(X, y)
        self.model = rf
        self.X_all = X
 
        result              = df.copy()
        result['Objective_Rating'] = cv_preds          # CV 預測，非訓練集預測
        result['Bias']      = y.values - cv_preds
 
        # (3)動態判定門檻（1.5σ）
        mean_b = result['Bias'].mean()
        std_b  = result['Bias'].std()
        upper  = mean_b + sigma_threshold * std_b
        lower  = mean_b - sigma_threshold * std_b
 
        result['Bias_Flag'] = result['Bias'].apply(
            lambda b: 'Suggest Lower'  if b > upper else
                      'Suggest Higher' if b < lower else 'Fair'
        )
 
        # 統計摘要，存起來給 run_bias_tests 統計檢定用
        cv_r2 = r2_score(y, cv_preds)
        print(f"✅ [分析完成] Residual Bias (Cross-Validated)")
        print(f"   CV R² = {cv_r2:.4f}  |  Bias 門檻 ±{sigma_threshold}σ = ±{std_b*sigma_threshold:.3f}")
        print(f"   Bias_Flag 分布：\n{result['Bias_Flag'].value_counts().to_string()}")
        return result

    # ── 2.統計偏誤檢定（針對敏感特徵進行 Bias 的 ANOVA / T-test 統計檢定）
    def run_bias_tests(self, result_df, original_df):
        """
        對每個敏感特徵做 T-test / ANOVA + Effect Size。
        這一步即為「群體層級的控制後偏誤分析」：比較的是殘差 Bias 而非原始評分，因此已排除客觀能力差異，與 _run_group_fairness 的結果平權互補。
        result_df  = engine.run() 回傳的結果。
        original_df = clean_df（含原始敏感特徵值，未被 LabelEncode）。
        """
        #(1)Cohen's d：衡量兩組平均數差異的效果量指標（effect size）
        def cohen_d(a, b):
            n1, n2 = len(a), len(b)
            if n1 < 2 or n2 < 2: return np.nan
            p = ((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1)) / (n1+n2-2)
            return (a.mean()-b.mean()) / np.sqrt(p) if p > 0 else 0.
 
        #(2)Eta：衡量自變量對依變量的貢獻程度 (SSB/SST) 
        def eta2(gs):
            all_ = np.concatenate([g.values for g in gs]); gm = all_.mean()
            SSb  = sum(len(g)*(g.mean()-gm)**2 for g in gs)  # 組間差
            SSt  = sum((x-gm)**2 for x in all_)  # 總差異
            return SSb/SSt if SSt > 0 else 0
 
        bias_series = result_df['Bias'].values
        test_results = []
 
        for col in self.sensitive_cols:
            if col not in original_df.columns: continue

            tmp = pd.DataFrame({
                'Bias': result_df['Bias'],
                'grp' : original_df[col].astype(str)
            }).dropna()
            tmp = tmp[~tmp['grp'].isin(['nan', 'unknown', 'none', 'NaN'])]
 
            grouped = [(name, g['Bias']) for name, g in tmp.groupby('grp') if len(g) >= 5]
            if len(grouped) < 2:
                continue
            vn, vg = [n for n, _ in grouped], [g for _, g in grouped]
            
            # 只有 2 個群體（如男女）使用 T-test、有 3 個以上群體（如多個種族）使用 One-way ANOVA
            if len(vg) == 2:
                _, p = stats.ttest_ind(*vg, nan_policy='omit')
                eff  = abs(cohen_d(vg[0], vg[1]))
                meth, eff_name = 'T-test', "Cohen's d"
            else: 
                _, p = stats.f_oneway(*vg) # One-way ANOVA
                eff  = eta2(vg)
                meth, eff_name = 'ANOVA', 'η²'
 
            sig = '🚨 顯著偏誤' if p < 0.05 else '✅ 無顯著差異'
            print(f"\n  ▶ {col}：{meth}  p={p:.4f}  {sig}  |  {eff_name}={eff:.4f}")
            for gn, gd in zip(vn, vg):
                print(f"      {str(gn):<25} mean={gd.mean():.3f}  (n={len(gd)})")
            test_results.append({
                '特徵': col, '方法': meth, 'p_value': round(p, 4),
                '顯著': p < 0.05, '效果量': round(eff, 4), '檢定名稱': eff_name
            })

 
        return pd.DataFrame(test_results)

    # ── 3.SHAP 解釋圖 ───────────────────────────────────────
    def generate_shap_summary(self):
        if self.model is None or self.X_all is None:
            raise ValueError("請先執行 run(method='residual') 來訓練模型。")
 
        try:
            import shap
        except ImportError:
            raise ImportError("請安裝 shap：pip install shap")
 
        plt.close('all')   # 清除殘留畫布
 
        explainer   = shap.TreeExplainer(self.model)
        shap_values = explainer.shap_values(self.X_all)
 
        shap.summary_plot(shap_values, self.X_all, show=False)
        fig = plt.gcf()
        fig.set_size_inches(10, 6)
        plt.tight_layout()
        return fig   


   # ── 4.Group Fairness 進行群體公平性分析 (Disparate Impact & SPD)
    def _run_group_fairness(self, df, high_perf_threshold=0.7,
                             privileged_group=None, sensitive_target=None):
        if sensitive_target is None:
            if self.sensitive_cols:
                sensitive_target = self.sensitive_cols[0]
                print(f"⚠️  sensitive_target 未指定，自動使用第一個敏感特徵：{sensitive_target}")
            else:
                raise ValueError("請傳入 sensitive_target 參數（例如 'gender'）")
 
        result_df = df.copy()

        # 二元化連續評分 (例如: 分數在前 30% 視為 High Performer)
        threshold_val = result_df[self.target_col].quantile(high_perf_threshold)
        result_df['Is_High_Performer'] = (result_df[self.target_col] >= threshold_val).astype(int)
 
        if sensitive_target not in result_df.columns:
            raise ValueError(f"找不到敏感特徵欄位：{sensitive_target}")
 
        # 計算各群體獲得高分的機率
        rates = result_df.groupby(sensitive_target)['Is_High_Performer'].mean()
        print(f"\n  各群體高績效比例：\n{rates.to_string()}")
 
        # 動態推斷：將「高分比例最高」的群體自動設為基準組 (Privileged Group)
        if privileged_group is None:
            privileged_group = rates.idxmax()
            print(f"  ⚠️  privileged_group 未指定，自動使用比例最高的群體：{privileged_group}")
 
        # normalize case
        rates.index = rates.index.str.lower() if hasattr(rates.index, 'str') else rates.index
        if privileged_group is not None:
            privileged_group = str(privileged_group).lower()
        if privileged_group not in rates:
            raise ValueError(f"找不到基準組 '{privileged_group}'，請確認資料中的實際類別值。")
 
        base_rate  = rates[privileged_group]
        # Disparate Impact (DI) = 弱勢組晉升率 / 優勢組晉升率 (< 0.8 通常視為有偏誤)           
        di_scores  = {g: (r/base_rate if base_rate > 0 else 0) for g, r in rates.items()}
        # Statistical Parity Difference (SPD) = 弱勢組晉升率 - 優勢組晉升率
        spd_scores = {g: r - base_rate for g, r in rates.items()}
        # r = P(高績效｜群體 g)
        # base_rate = P(高績效｜基準群體)
 
        print(f"✅ [分析完成] Group Fairness｜基準組：{privileged_group}")
        return {
            "Disparate_Impact"              : di_scores,
            "Statistical_Parity_Difference" : spd_scores,
            "High_Performer_Rates"          : rates.to_dict()
        }